## EDA

Exploratory analysis that motivates the modelling choices in the price-imitator notebook.

**Sequence**
1. Load + shape
2. Grain validation
3. Market-group structure
4. Data-quality checks
5. Feature diagnostics
6. Seasonal structure
7. Target definition

## 1. Load + shape

Load csv-file, parse the arrival week, and confirm the table size before any filtering.

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

In [3]:
CSV = Path(r"C:\Users\FrederikS\Documents\ECG_better\data\raw\French_Riviera_Multiple_campsites_datasets_capacity_over_200_anonymised.csv")

df = pd.read_csv(CSV)
df["WeekStartDate"] = pd.to_datetime(df["WeekStartDate"])
df["Year"] = df["WeekStartDate"].dt.year

print(f"rows: {len(df):,} | columns: {df.shape[1]}")

C:\Users\FrederikS\AppData\Local\Temp\ipykernel_54480\234492281.py:3: DtypeWarning: Columns (19,22,23,24,25,26,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV)


rows: 1,214,072 | columns: 41


In [4]:
# What does the data look like?
df.head()

,ReservableOptionId,ReservableOptionMarketGroupId,BrandGroupCode,CampsiteCode,AccoTypeRangeCode,MarketGroupCode,WeekBeforeArrival,WeekStartDate,ArrivalMonth,SpecialPeriodCode,...,AverageDiscPriceForWeekBeforeArrivalLastYear,HistoricalBookedNights,HistoricalBookedNightsLastYear,CumulativeHistoricalBookedNights,CumulativeHistoricalBookedNightsLastYear,TotalBookedNights,CapacityLastYear,Capacity,SoldOutWeekBeforeArrival,Year
0,82307070,858803906,BrandGroup1,Campsite021,MH | Comfort+,CN,78,2025-07-19,7,NaN,...,0.0,NaN,0,0,0,172,266.0,266,NaN,2025
1,82307070,858803906,BrandGroup1,Campsite021,MH | Comfort+,CN,77,2025-07-19,7,NaN,...,0.0,NaN,0,0,0,172,266.0,266,NaN,2025
2,82307070,858803906,BrandGroup1,Campsite021,MH | Comfort+,CN,76,2025-07-19,7,NaN,...,0.0,NaN,0,0,0,172,266.0,266,NaN,2025
3,82307070,858803906,BrandGroup1,Campsite021,MH | Comfort+,CN,75,2025-07-19,7,NaN,...,0.0,NaN,0,0,0,172,266.0,266,NaN,2025
4,82307070,858803906,BrandGroup1,Campsite021,MH | Comfort+,CN,74,2025-07-19,7,NaN,...,0.0,NaN,0,0,0,172,266.0,266,NaN,2025


In [5]:
# Which columns are available?
df.columns

Index(['ReservableOptionId', 'ReservableOptionMarketGroupId', 'BrandGroupCode',
       'CampsiteCode', 'AccoTypeRangeCode', 'MarketGroupCode',
       'WeekBeforeArrival', 'WeekStartDate', 'ArrivalMonth',
       'SpecialPeriodCode', 'SeasonalCluster', 'CampsiteCluster',
       'CampsiteCountry', 'CampsiteRegion', 'CampsiteType', 'CampsiteLatitude',
       'CampsiteLongitude', 'AccommodationType', 'AccommodationRange', 'Airco',
       'Bedrooms', 'DeckingType', 'HotTub', 'Tropical', 'Roof', 'Kitchen',
       'DeckingExtras', 'Bathrooms', 'Sleeps', 'TV',
       'AverageDiscPriceForWeekBeforeArrival',
       'AverageDiscPriceForWeekBeforeArrivalLastYear',
       'HistoricalBookedNights', 'HistoricalBookedNightsLastYear',
       'CumulativeHistoricalBookedNights',
       'CumulativeHistoricalBookedNightsLastYear', 'TotalBookedNights',
       'CapacityLastYear', 'Capacity', 'SoldOutWeekBeforeArrival', 'Year'],
      dtype='object')

## 2. Grain validation

`ReservableOptionId` should map one-to-one onto an arrival week. If the count of unique
ids equals the count of unique `(ReservableOptionId, WeekStartDate)` pairs, then
**one option = one arrival week** and the option id alone identifies the arrival week.

In [6]:
# ReservableOptionId is a unique combo of accommodation type and arrival week.
unique_option_id = df["ReservableOptionId"].nunique()
unique_tuples = df[["ReservableOptionId", "WeekStartDate"]].drop_duplicates().shape[0]

print(f"unique ReservableOptionId            : {unique_option_id:,}")
print(f"unique (ReservableOptionId, WeekStart): {unique_tuples:,}")

if unique_option_id == unique_tuples:
    print("=> one option = one arrival week")

unique ReservableOptionId            : 3,842
unique (ReservableOptionId, WeekStart): 3,842
=> one option = one arrival week


## 3. Market-group structure

Each option carries **4 market groups**, each with its own bookings. The price-initialiser
works at the option level, so the bookings have to be aggregated up from the market groups
to a single option-week record.

In [7]:
# How many market groups does each option carry?
market_groups_per_option = df.groupby("ReservableOptionId")["MarketGroupCode"].nunique()
print(market_groups_per_option.value_counts())
print("\nmarket group codes:", sorted(df["MarketGroupCode"].unique()))

MarketGroupCode
4    3842
Name: count, dtype: int64

market group codes: ['CN', 'IR', 'NK', 'US']


In [8]:
# Aggregate the 4 market groups up to one option-week record.
df["OptionCumulativeBookedNights"] = (
    df.groupby("ReservableOptionId")["CumulativeHistoricalBookedNights"].transform("sum")
)

option_week = df.drop_duplicates(["ReservableOptionId", "WeekBeforeArrival"]).copy()
option_week["BookingsPercentage"] = (
    option_week["OptionCumulativeBookedNights"] / option_week["Capacity"]
)

print(f"raw rows          : {len(df):,}")
print(f"option-week rows  : {len(option_week):,}")

raw rows          : 1,214,072
option-week rows  : 303,518


## 4. Data-quality checks

- **Monotonicity** — `CumulativeHistoricalBookedNightss` must never decrease as the arrival date approaches.
- **Overbooking** — flag rows where `TotalBookedNights > Capacity`.
- **LastYear missingness** — distinguish genuine prior-year data from placeholder fills.

In [20]:
# CumulativeHistoricalBookedNights should never drop as the horizon shrinks (far -> near).
keys = ["ReservableOptionId", "MarketGroupCode", "WeekStartDate"]

ordered = df.sort_values(keys + ["WeekBeforeArrival"], ascending=[True, True, True, False])
ordered["booking_diff"] = ordered.groupby(keys)["CumulativeHistoricalBookedNights"].diff()

violations = ordered.loc[ordered["booking_diff"] < 0, keys].drop_duplicates()
print(f"series with a decrease in CumulativeHistoricalBookedNights: {len(violations)}")

series with a decrease in CumulativeHistoricalBookedNights: 0


In [10]:
# Overbooking: rows where bookings exceed capacity.
overbooked = df[df["TotalBookedNights"] > df["Capacity"]].copy()
overbooked["OverbookedPct"] = (
    (overbooked["TotalBookedNights"] - overbooked["Capacity"]) / overbooked["Capacity"] * 100
)

print(f"overbooked rows: {len(overbooked):,} ({len(overbooked) / len(df):.2%} of all rows)")
overbooked.groupby("ReservableOptionId")["OverbookedPct"].max().sort_values(ascending=False).head()

overbooked rows: 948 (0.08% of all rows)


ReservableOptionId
82155090    23.853211
82155031    21.338156
82155034     5.786618
82155027     2.486679
82155028     2.040816
Name: OverbookedPct, dtype: float64

In [11]:
# Prior-year columns: how complete are they per season year?
ly = [
    "CumulativeHistoricalBookedNightsLastYear",
    "AverageDiscPriceForWeekBeforeArrivalLastYear",
    "CapacityLastYear",
]
df[ly].notna().groupby(df["Year"]).mean()

,CumulativeHistoricalBookedNightsLastYear,AverageDiscPriceForWeekBeforeArrivalLastYear,CapacityLastYear
Year,,,
2024,1.0,1.0,0.000000
2025,1.0,1.0,0.702786
2026,1.0,1.0,0.814921


In [12]:
# Placeholder vs real: a constant / all-zero spread signals a fake fill,
# a genuine spread signals real prior-year booking data.
df.loc[df["Year"] == 2024, "AverageDiscPriceForWeekBeforeArrivalLastYear"].describe()

count    483480.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: AverageDiscPriceForWeekBeforeArrivalLastYear, dtype: float64

## 5. Feature diagnostics

- **Categorical cardinality** — flags high-cardinality fields (e.g. `CampsiteCode`) that
  blow up one-hot width and destabilise Ridge.
- **Boolean constancy within `AccoTypeRangeCode`** — if an attribute never varies inside a
  type|range group (`max == 1`), it is redundant with the `AccommodationType` + `RangeType`
  encoding; `max >= 2` means it carries independent signal and is worth keeping.

In [13]:
# Categorical cardinality — high-cardinality fields are one-hot-width risks.
for column in df.select_dtypes("object").columns:
    n = df[column].nunique()
    print(f"{column}: {n} unique")
    if n <= 12:
        print("   ", sorted(df[column].dropna().unique()))

BrandGroupCode: 2 unique
    ['BrandGroup1', 'BrandGroup2']
CampsiteCode: 37 unique
AccoTypeRangeCode: 8 unique
    ['Chalet | Comfort', 'MH | Budget', 'MH | Budget+', 'MH | Comfort', 'MH | Comfort+', 'MH | Premium', 'MH | Premium+', 'MH | Ultimate']
MarketGroupCode: 4 unique
    ['CN', 'IR', 'NK', 'US']
SeasonalCluster: 5 unique
    ['HS', 'LS', 'S1', 'S2', 'WTR']
CampsiteCluster: 5 unique
    ['A', 'A-', 'B', 'B+', 'C']
CampsiteCountry: 1 unique
    ['France']
CampsiteRegion: 2 unique
    ['Occitanie', "Provence-Alpes-Côte d'Azur"]
CampsiteType: 2 unique
    ['Own', 'Partner']
AccommodationType: 2 unique
    ['Chalet', 'MH']
AccommodationRange: 7 unique
    ['Budget', 'Budget+', 'Comfort', 'Comfort+', 'Premium', 'Premium+', 'Ultimate']
Airco: 2 unique
    [False, True]
Bedrooms: 2 unique
    ['2 Bed', '3 Bed']
DeckingType: 4 unique
    ['Covered Decking', 'Decking', 'Lounge Decking', 'Small Decking']
HotTub: 1 unique
    [False]
Tropical: 1 unique
    [False]
Roof: 1 unique
    [Fals

In [ ]:
# How many unique options does each campsite have? (37 campsites have >100 unique options.)

unique_counts = (
    df.groupby("CampsiteCode")["ReservableOptionId"]
    .nunique()
    .reset_index(name="UniqueReservableOptions")
)
unique_counts.sort_values("UniqueReservableOptions", ascending=False).head(37)

,CampsiteCode,UniqueReservableOptions
18,Campsite019,500
16,Campsite017,400
29,Campsite030,379
2,Campsite003,281
17,Campsite018,233
6,Campsite007,194
12,Campsite013,179
1,Campsite002,175
10,Campsite011,161
20,Campsite021,139


In [14]:
# Are these attributes fully determined by AccoTypeRangeCode?
redundancy_candidates = ["Airco", "TV", "DeckingType", "Bathrooms"]

determinism = (
    df.groupby("AccoTypeRangeCode")[redundancy_candidates]
    .nunique()
    .max()                       # worst case across all type|range groups
    .rename("max_unique_within_group")
)
print(determinism)
# max == 1  -> redundant: deterministic function of AccoTypeRangeCode
# max >= 2  -> keep: varies within a type|range, so it adds information

Airco          2
TV             2
DeckingType    3
Bathrooms      2
Name: max_unique_within_group, dtype: int64


## 6. Seasonal structure

- **`Season_pivot`** — confirms every `SeasonalCluster` appears in **both** season years.
  The model trains on 2024 and tests on 2025, and the segment baseline maps 2025 options
  onto 2024 segment means; a cluster missing from either year would have no cross-year
  support and silently fall back to the global mean.
- **Occupancy distributions** per `SeasonalCluster × AccommodationRange` — the fill-rate
  shape differs sharply by season, so a single global target is inadequate: **targets must
  be defined per season.**

In [15]:
# Which SeasonalClusters appear in each season year, and how many options back them?
season_pivot = (
    df.groupby(["SeasonalCluster", "Year"])
    .agg(
        MinDate=("WeekStartDate", "min"),
        MaxDate=("WeekStartDate", "max"),
        UniqueOptions=("ReservableOptionId", "nunique"),
    )
    .reset_index()
    .sort_values(["SeasonalCluster", "Year"])
)
print(season_pivot.to_string(index=False))

SeasonalCluster  Year    MinDate    MaxDate  UniqueOptions
             HS  2024 2024-07-06 2024-08-24            781
             HS  2025 2025-07-05 2025-08-23            556
             LS  2024 2024-06-08 2024-06-15            167
             LS  2025 2025-04-05 2025-06-14            606
             LS  2026 2026-03-28 2026-05-30            697
             S1  2024 2024-06-22 2024-06-29            172
             S1  2025 2025-06-21 2025-06-28            127
             S2  2024 2024-08-31 2024-09-28            330
             S2  2025 2025-08-30 2025-09-27            272
            WTR  2024 2024-10-05 2024-10-26             80
            WTR  2025 2025-10-04 2025-10-25             54


In [ ]:
# Capacity-weighted booking fill per campsite across arrival weeks (at window open, WBA == 0).


at_start = df[df["WeekBeforeArrival"] == 0].copy()
at_start["OptionBookedNights"] = (
    at_start.groupby("ReservableOptionId")["CumulativeHistoricalBookedNights"].transform("sum")
)
at_start_option = at_start.drop_duplicates("ReservableOptionId")

campsite_weighted = at_start_option.groupby(["WeekStartDate", "CampsiteCode"]).agg(
    booked=("OptionBookedNights", "sum"),
    capacity=("Capacity", "sum"),
)
campsite_table_weighted = (campsite_weighted["booked"] / campsite_weighted["capacity"]).unstack()

campsite_dropdown = widgets.Dropdown(
    options=["All"] + list(campsite_table_weighted.columns),
    value="All",
    description="Campsite:",
)
plot_output = widgets.Output()


def _on_change(change):
    if change["name"] != "value" or change["new"] is None:
        return
    with plot_output:
        plot_output.clear_output(wait=True)
        selection = change["new"]
        data = (
            campsite_table_weighted
            if selection == "All"
            else campsite_table_weighted[[selection]]
        )
        ax = data.plot(figsize=(12, 6))
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
        ax.set_ylabel("Capacity-weighted bookings %")
        plt.tight_layout()
        plt.show()


campsite_dropdown.observe(_on_change, names="value")
display(campsite_dropdown, plot_output)
_on_change({"name": "value", "new": campsite_dropdown.value})

Dropdown(description='Campsite:', options=('All', 'Campsite001', 'Campsite002', 'Campsite003', 'Campsite004', …

Output()

## 7. Target definition

Two candidate targets were considered. Both collapse the option × horizon panel to **one
target per reservable option**, but they answer different questions.

**A — Demand-weighted opening price (the target the model uses).**
Collapse the 4 market groups into a single demand-weighted price per option × horizon
(weight = `CumulativeHistoricalBookedNights`), then read the price at the **opening of the
booking window** (largest `WeekBeforeArrival` with real demand). This is a *price imitator*:
it learns to reproduce the price ECG historically opened with, given the option's
characteristics and demand history.

**B — `TargetLabeler` good-price subset (TBC).**
Label each option-week as `underpriced` / `on_target` / `overpriced` from how it sold
(sold out near arrival, or filled to its segment's occupancy bar), then keep **only the
`on_target` rows** as training labels. This is a *price initializer*: it learns only from
prices that demonstrably produced good outcomes, deliberately ignoring historical prices
that over- or under-sold.

**Choice — the model uses A (weighted opening price)** because it is well-defined for every
option, needs no occupancy-threshold dials, and gives a dense, leakage-free target for a
first interpretable baseline. B is the natural next step once we want the model to *correct*
historical pricing rather than replicate it, but it discards most rows and depends on the
sell-out / occupancy-bar assumptions verified in sections 4 and 6.

In [ ]:

per_horizon = (
    df.assign(_weighted=df["AverageDiscPriceForWeekBeforeArrival"]
              * df["CumulativeHistoricalBookedNights"])
    .groupby(["ReservableOptionId", "WeekBeforeArrival"], as_index=False)
    .agg(booked=("CumulativeHistoricalBookedNights", "sum"),
         weighted=("_weighted", "sum"))
)
per_horizon["WeightedPrice"] = np.where(
    per_horizon["booked"] > 0,
    per_horizon["weighted"] / per_horizon["booked"],
    np.nan,
)

valid = per_horizon[per_horizon["WeightedPrice"].notna()]
opening_idx = valid.groupby("ReservableOptionId")["WeekBeforeArrival"].idxmax()
weighted_opening_price = (
    valid.loc[opening_idx, ["ReservableOptionId", "WeightedPrice"]]
    .rename(columns={"WeightedPrice": "InitialPrice"})
    .reset_index(drop=True)
)

print(f"Candidate A rows (one per option): {len(weighted_opening_price):,}")
print(weighted_opening_price["InitialPrice"].describe().round(2).to_string())
display(weighted_opening_price.head())

Candidate A rows (one per option): 3,306
count    3306.00
mean       91.27
std        83.57
min         0.00
25%        42.00
50%        59.00
75%       131.00
max       538.86


,ReservableOptionId,InitialPrice
0,82074122,50.000000
1,82074123,0.000000
2,82074124,175.000000
3,82074125,0.000000
4,82074126,49.714286
